[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ironyr/llm-project/blob/feat/v1/notebooks/03_retrieval_benchmark.ipynb)

**Google Colab:** Set `GIT_REPO`, run bootstrap + ingest. Retrieval uses the on-disk Qdrant store built by `ingest.py` (under `data/qdrant_store/`).

In [ ]:
# === Colab / local bootstrap (run first) ===
import os
import subprocess
import sys
from pathlib import Path

# Colab: set to your fork’s HTTPS clone URL (must match the repo that hosts this notebook)
GIT_REPO = "https://github.com/IronYR/llm-project.git"
GIT_BRANCH = "feat/v1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.chdir("/content")
    root = Path("/content/llm-project")
    if not (root / "config.yaml").is_file():
        if "YOUR_GITHUB_USERNAME" in GIT_REPO:
            raise RuntimeError(
                "Edit GIT_REPO to your GitHub fork, e.g. https://github.com/<you>/llm-project.git"
            )
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, str(root)],
            check=True,
        )
    os.chdir(root)
else:
    p = Path.cwd().resolve()
    if p.name == "notebooks":
        os.chdir(p.parent)
    elif not (p / "config.yaml").is_file() and (p.parent / "config.yaml").is_file():
        os.chdir(p.parent)

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

if IN_COLAB:
    from src.colab_setup import colab_pip_install
    colab_pip_install(ROOT)

print("ROOT =", ROOT)


In [ ]:
from src.colab_setup import ensure_knowledge_base

ensure_knowledge_base(ROOT)

# Retrieval benchmark (top-k)

Notebook version of `python evaluation/run_eval.py`, similar to [Bechmarking.ipynb](https://github.com/geetu040/nust-bank-chatbot/blob/main/notebooks/Bechmarking.ipynb) — but metrics are **retrieval match rates** on your **Qdrant + sentence-transformers** index (not ROUGE on generative answers).

Run after `python ingest.py` so the index exists.

In [ ]:
import json
from pathlib import Path

import yaml

from src.embeddings import EmbeddingIndex

cfg_path = ROOT / "config.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

data_cfg = cfg["data"]
kb_path = Path(data_cfg["processed_dir"]) / data_cfg["knowledge_base_file"]
with open(kb_path, encoding="utf-8") as f:
    kb = json.load(f)
docs = [d for d in kb.get("documents", []) if (d.get("question") or "").strip()]

index = EmbeddingIndex(cfg)
print(f"Evaluating {len(docs)} questions with stored answers")

In [ ]:
def eval_topk(cfg, docs, top_k: int):
    hits_1 = hits_k = n = 0
    for doc in docs:
        q = doc.get("question", "").strip()
        gold_a = (doc.get("answer") or "").strip()
        n += 1
        results = index.search(q, top_k=top_k)
        if not results:
            continue
        top = results[0]
        if top.get("answer", "").strip() == gold_a or top.get("question", "").strip() == q:
            hits_1 += 1
        if any(
            r.get("answer", "").strip() == gold_a or r.get("question", "").strip() == q
            for r in results
        ):
            hits_k += 1
    return n, (hits_1 / n if n else 0), (hits_k / n if n else 0)

rows = []
for k in (1, 3, 5, cfg["retrieval"]["top_k"]):
    n, r1, rk = eval_topk(cfg, docs, top_k=k)
    rows.append({"top_k": k, "n": n, "top1_match": round(r1, 4), "topk_match": round(rk, 4)})

import pandas as pd
pd.DataFrame(rows)